# DDS-Edge: xử lý mất cân bằng lớp ở tầng dữ liệu (Colab)

**Câu hỏi của notebook này:** lớp `phone` chiếm 41,6% ảnh train còn `talk_passenger` chỉ 8,4%. Nếu ta **sửa dữ liệu train** (thay vì sửa hàm loss như `class_weight`) thì mô hình có cải thiện, nhất là với các lớp ít ảnh không?

**Điểm mấu chốt cần hiểu trước khi làm:** mất cân bằng *không có sẵn trong dữ liệu gốc*. Mười lớp gốc c0–c9 của State Farm gần như bằng nhau (khoảng 1.200 đến 1.500 ảnh mỗi lớp trong tập train). Nó chỉ xuất hiện khi ta **gộp** c1+c2+c3+c4 thành `phone`. Vì vậy có hai hướng: (1) đổi cách *dùng nhãn*, (2) đổi *số ảnh mỗi lớp*.

**Các cách thử (đều chỉ đổi tập train; val và test giữ nguyên để so sánh công bằng):**

| Tên run | Cách làm | Ý tưởng |
|---|---|---|
| `effl0_cap` | **Giảm mẫu** lớp đông (`phone`, `reach_grooming` còn 2.000 ảnh) | Bớt ảnh lặp của lớp lớn |
| `effl0_aug` | **Tăng mẫu** lớp ít bằng ảnh augment lưu thật đến 2.000 ảnh | Cho lớp nhỏ nhiều mẫu hơn |
| `effl0_both` | Cả hai: **mọi lớp đúng 2.000 ảnh** | Cân bằng hoàn toàn |
| `effl0_fine10` | **Train trên 10 lớp gốc, chấm trên 6 nhóm** (cộng xác suất) | Không bỏ/tạo ảnh nào, dùng nhãn đã cân bằng sẵn |

Mô hình cố định là **EfficientNet-Lite0** (mô hình bạn chọn cho edge), giao thức giống mọi lần trước (10 epoch, AdamW, seed 42). Mốc để so: `effl0_clean` (cùng máy Colab): test macro-F1 **0,815**, F1 `talk_passenger` **0,416**.

**Cách dùng:** chạy từng cell từ trên xuống. Mỗi cell có phần giải thích đứng ngay trước nó. Tổng thời gian khoảng 40 đến 50 phút (4 lần train, mỗi lần 6 đến 10 phút, ước tính). **Cần `git push` code mới lên GitHub trước** (cell 3 chạy `git pull`).

## A. Chuẩn bị môi trường (cell 1 đến 5, giống các notebook trước)

### Cell 1: kiểm tra GPU
Phải thấy tên GPU (thường là `Tesla T4`). Nếu lỗi: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Bat GPU: Runtime -> Change runtime type -> T4 GPU'
print(torch.cuda.get_device_name(0))

### Cell 2: nối Google Drive
Drive dùng để đọc file dữ liệu `sf_320x240.zip` và để **lưu kết quả train** (không mất khi Colab ngắt).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Cell 3: tải code từ GitHub và cài thư viện
`git clone` lần đầu, các lần sau `git pull`. Dòng cuối in ra commit đang dùng. Code mới của notebook này (`balance.py`, cờ `--eval_grouping`) **chỉ có sau khi bạn push**; nếu chưa push, các lệnh bên dưới báo `No module named ...balance` hoặc `unrecognized arguments: --eval_grouping`.

In [ ]:
%cd /content
!git clone https://github.com/ColorOfDreams/dds-edge.git 2>/dev/null || echo 'already cloned'
%cd /content/dds-edge
!git pull -q
!pip install -q timm onnx onnxruntime tabulate matplotlib
!pip install -q -e . --no-deps
!git log --oneline -1

### Cell 4: giải nén dữ liệu và nối vào đúng chỗ
Giải nén `sf_320x240.zip` vào ổ đĩa của Colab rồi tạo liên kết `data/processed/sf_320x240`. Thư mục này **ghi được**, nên các ảnh augment mới (cell 8) sẽ được lưu ngay vào đó.

**Kết quả cần thấy:** `20007 images; {'test': 4, 'train': 18, 'val': 4} drivers per split`.

In [ ]:
# data: extract to the local disk (much faster than reading from Drive), then link to data/processed/sf_320x240
# accepts sf_320x240*.zip (preferred) or sf_320x240*.rar in My Drive
import os, glob, shutil, zipfile, subprocess
os.chdir('/content/dds-edge')
if not glob.glob('/content/sf_data/**/manifest.csv', recursive=True):
    arch = None
    for ext in ('zip', 'rar'):
        found = (glob.glob(f'/content/drive/MyDrive/sf_320x240*.{ext}')
                 or glob.glob(f'/content/drive/MyDrive/**/sf_320x240*.{ext}', recursive=True))
        if found:
            arch = found[0]
            break
    assert arch, 'Khong thay sf_320x240*.zip/.rar trong My Drive: da upload len Drive chua?'
    print('archive:', arch)
    shutil.rmtree('/content/sf_data', ignore_errors=True)
    os.makedirs('/content/sf_data')
    if arch.endswith('.zip'):
        with zipfile.ZipFile(arch) as z:
            for info in z.infolist():
                name = info.filename.replace(chr(92), '/')  # zips made on Windows use backslashes
                if name.endswith('/'):
                    continue
                dst = os.path.join('/content/sf_data', name)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                with z.open(info) as f, open(dst, 'wb') as g:
                    shutil.copyfileobj(f, g)
    else:  # .rar needs the unrar tool
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'unrar'], check=True)
        subprocess.run(['unrar', 'x', '-idq', '-y', arch, '/content/sf_data/'], check=True)
hits = glob.glob('/content/sf_data/**/manifest.csv', recursive=True)
assert hits, 'Khong thay manifest.csv sau khi giai nen: ' + str(os.listdir('/content/sf_data')[:5])
src = os.path.dirname(hits[0])
os.makedirs('data/processed', exist_ok=True)
link = 'data/processed/sf_320x240'
if os.path.islink(link):  # stale link from an earlier run
    os.unlink(link)
assert not os.path.exists(link), f'{link} is a real folder, remove it first'
os.symlink(src, link)
import pandas as pd
m = pd.read_csv(link + '/manifest.csv')
print(len(m), 'images;', m.groupby('split')['group'].nunique().to_dict(), 'drivers per split')  # expect 20007; test 4, train 18, val 4

### Cell 5: lưu kết quả train vào Drive
Biến `runs/` thành liên kết tới `MyDrive/dds-edge-runs/`. Mỗi lần train xong, `best.pt`, `metrics.json`, `predictions.csv`… nằm sẵn trong Drive.

In [ ]:
# write runs/ straight to Drive so results survive a disconnect
%cd /content/dds-edge
!mkdir -p /content/drive/MyDrive/dds-edge-runs
!rm -rf runs
!ln -s /content/drive/MyDrive/dds-edge-runs runs
!ls -la runs | head -3

## B. Nhìn vào mất cân bằng

### Cell 6: đếm ảnh theo lớp, trước và sau khi gộp nhãn
In hai bảng và vẽ biểu đồ cho **tập train**:
- **6 nhóm** (cách ta đang dùng): `phone` áp đảo, tỉ lệ lớn nhất so với nhỏ nhất khoảng **4,9**.
- **10 lớp gốc** (c0–c9): gần bằng nhau, tỉ lệ khoảng **1,3**.

**Điều cần rút ra:** mất cân bằng là hệ quả của việc gộp nhãn, không phải của dữ liệu. Đây là lý do cách `effl0_fine10` (train trên 10 lớp) đáng thử.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

m = pd.read_csv('/content/dds-edge/data/processed/sf_320x240/manifest.csv')
tr = m[m.split == 'train']
six = tr.label.value_counts()
ten = tr.raw_label.value_counts().sort_index()
print('TRAIN, 6 nhóm:'); print(six.to_string(), f'\n  tỉ lệ lớn/nhỏ = {six.max() / six.min():.2f}\n')
print('TRAIN, 10 lớp gốc:'); print(ten.to_string(), f'\n  tỉ lệ lớn/nhỏ = {ten.max() / ten.min():.2f}')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
six.plot.bar(ax=ax[0], color='#00A95C'); ax[0].set_title('6 nhóm (lệch)')
ten.plot.bar(ax=ax[1], color='#F58643'); ax[1].set_title('10 lớp gốc (cân bằng)')
for a in ax: a.set_ylabel('số ảnh train')
plt.tight_layout(); plt.show()

## C. Tạo dữ liệu cân bằng

### Cell 7: chọn mức cân bằng `TARGET`
`TARGET = 2000` nghĩa là mỗi nhóm sẽ có khoảng 2.000 ảnh train. Chọn 2.000 vì: `phone` (5.738) và `reach_grooming` (2.653) bị cắt xuống, còn `normal` (1.276) và `talk_passenger` (1.161) được tăng lên, và 2.000 không quá xa mọi lớp (đỡ phải tạo quá nhiều ảnh giả hay bỏ quá nhiều ảnh thật).

In [ ]:
TARGET = 2000
print('TARGET =', TARGET)

### Cell 8: tạo ba bộ dữ liệu cân bằng bằng `balance.py`
Lệnh sinh ra 3 file manifest mới cạnh `manifest.csv`:
- `manifest_cap.csv`: **giảm mẫu**. Mỗi nhóm giữ tối đa `TARGET` ảnh, lấy **luân phiên giữa các tài xế** (để không tài xế nào biến mất).
- `manifest_aug.csv`: **tăng mẫu**. Nhóm nào dưới `TARGET` được thêm ảnh augment (cắt ngẫu nhiên, xoay ±8°, đổi sáng/màu, **không lật ngang**) từ chính ảnh train của nhóm đó, lưu thành file JPEG thật trong `images/aug/`.
- `manifest_both.csv`: cả hai, mọi nhóm đúng `TARGET` ảnh.

Chương trình tự kiểm tra: **val và test không đổi một dòng nào**, không có đường dẫn trùng, mỗi tài xế vẫn chỉ ở một tập. Với mỗi chế độ in bảng "trước → sau" và tỉ lệ lớn/nhỏ. Khoảng 1 đến 2 phút.

**Cần thấy:** `cap`: tỉ lệ còn khoảng 1,7; `aug`: khoảng 2,9; `both`: đúng 1,00. Ảnh augment nằm trong ổ đĩa tạm của Colab, mất khi phiên đóng (chạy lại cell này là có lại).

In [ ]:
%cd /content/dds-edge
!python -m dds_edge.dl.balance --mode cap  --target {TARGET}
!python -m dds_edge.dl.balance --mode aug  --target {TARGET}
!python -m dds_edge.dl.balance --mode both --target {TARGET}

### Cell 9: xem ảnh augment bằng mắt
Hàng trên là **ảnh gốc**, hàng dưới là **bản augment** của chính ảnh đó (lớp `talk_passenger`). Bạn kiểm tra: ảnh vẫn nhận ra hành vi, không bị méo quá mức, không bị lật. Nếu ảnh augment trông vô lý thì cách `aug` sẽ làm hại mô hình.

In [ ]:
import pandas as pd
from PIL import Image
from IPython.display import display

m = pd.read_csv('/content/dds-edge/data/processed/sf_320x240/manifest_aug.csv')
a = m[m.path.str.contains('/aug/aug/') & (m.label == 'talk_passenger')].sample(4, random_state=1)
canvas = Image.new('RGB', (4 * 320, 2 * 240))
for i, (_, r) in enumerate(a.iterrows()):
    stem = r.path.split('/')[-1].rsplit('_aug', 1)[0]
    orig = m[m.path.str.endswith('/' + stem + '.jpg') & ~m.path.str.contains('/aug/')].iloc[0].path
    canvas.paste(Image.open('/content/dds-edge/data/' + orig).convert('RGB'), (i * 320, 0))
    canvas.paste(Image.open('/content/dds-edge/data/' + r.path).convert('RGB'), (i * 320, 240))
display(canvas)

## D. Chạy các thí nghiệm (EfficientNet-Lite0, 10 epoch)

Mỗi cell dưới đây là **một lần train**. Dán thẳng lệnh, không dùng biến, để tránh lỗi tên run. Sau mỗi lần xong, nhìn dòng JSON cuối: `n_params` phải là **3378694** (đúng EfficientNet-Lite0), rồi ghi lại `test_macro_f1` và F1 của `talk_passenger`.

**Quy tắc đọc kết quả:** chênh lệch dưới khoảng **0,03** so với mốc 0,815 *chưa chắc là thật* (đó là mức nhiễu giữa các lần chạy). Chỉ kết luận khi chênh lớn hơn, hoặc khi nhiều cách cùng chỉ về một hướng.

### Cell 10: (tuỳ chọn) chạy lại mốc `effl0_clean`
Chỉ cần nếu thư mục `effl0_clean` chưa có trong Drive. Giữ `#` nếu đã có kết quả (0,815).

In [ ]:
%cd /content/dds-edge
# !python -u -m dds_edge.dl.train --arch efficientnet_lite0 --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --workers 2 --name effl0_clean

### Cell 11: `effl0_cap` (giảm mẫu lớp đông)
Train trên `manifest_cap.csv` (khoảng 9.400 ảnh train, ít hơn nên nhanh hơn, khoảng 6 phút).
**Xem gì:** `phone` bị bỏ bớt ảnh nhưng vẫn nhiều ví dụ (2.000), nên F1 của `phone` có thể giữ nguyên; câu hỏi là `normal` và `talk_passenger` có tăng không.

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --arch efficientnet_lite0 --grouping grouped6 --manifest processed/sf_320x240/manifest_cap.csv --workers 2 --name effl0_cap

### Cell 12: `effl0_aug` (tăng mẫu lớp ít bằng ảnh augment)
Train trên `manifest_aug.csv` (khoảng 16.400 ảnh, nhiều nhất nên lâu nhất, khoảng 10 phút).
**Xem gì:** ảnh augment là bản biến đổi của ảnh có sẵn, không thêm thông tin thật, nên đừng kỳ vọng nhiều; nhưng nó đưa các lớp nhỏ lên cùng mức nhìn thấy trong mỗi epoch.

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --arch efficientnet_lite0 --grouping grouped6 --manifest processed/sf_320x240/manifest_aug.csv --workers 2 --name effl0_aug

### Cell 13: `effl0_both` (cả hai, mọi lớp đúng 2.000 ảnh)
Train trên `manifest_both.csv` (12.000 ảnh, khoảng 7 phút). Đây là cách cân bằng hoàn toàn nhất.

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --arch efficientnet_lite0 --grouping grouped6 --manifest processed/sf_320x240/manifest_both.csv --workers 2 --name effl0_both

### Cell 14: `effl0_fine10` (train 10 lớp gốc, chấm 6 nhóm)
Khác ba cách trên ở chỗ **không đổi số ảnh nào**. Mô hình học phân biệt 10 lớp c0–c9 (đã cân bằng, và học được cả sự khác nhau giữa nhắn tin tay trái và tay phải), còn lúc chấm điểm ta **cộng xác suất** các lớp con thành 6 nhóm (ví dụ `phone` = c1+c2+c3+c4). Cờ làm việc này: `--grouping full10 --eval_grouping grouped6`. Điểm in ra và lưu (`val_macro_f1`, `test_macro_f1`) đã là điểm của **6 nhóm** nên so được trực tiếp với các lần khác; ngoài ra `metrics.json` còn lưu điểm 10 lớp (`test_f1_per_class_fine`).

Khoảng 8 phút.

In [ ]:
%cd /content/dds-edge
!python -u -m dds_edge.dl.train --arch efficientnet_lite0 --grouping full10 --eval_grouping grouped6 --manifest processed/sf_320x240/manifest.csv --workers 2 --name effl0_fine10

## E. So sánh kết quả

### Cell 15: bảng so sánh mọi lần chạy
Đọc `metrics.json` của mốc `effl0_clean`, hai lần xử lý cũ ở mức hàm loss (`effl0_cw_sqrt`, `effl0_smp_sqrt`) và bốn lần mới, rồi in bảng sắp theo test macro-F1:
- **val F1** (chọn epoch), **test acc**, **test macro-F1** (điểm chính),
- **chênh so với mốc** (đỏ/xanh nằm trong ±0,03 thì chưa kết luận được),
- F1 của `normal` và `talk_passenger` (hai lớp yếu), và **khoảng cách val − test** (càng lớn càng đáng ngờ).

Run nào chưa chạy hoặc thiếu file sẽ ghi `THIẾU`, không làm cell lỗi.

In [ ]:
import json, os
RUNS = '/content/dds-edge/runs'
names = ['effl0_clean', 'effl0_cw_sqrt', 'effl0_smp_sqrt', 'effl0_cap', 'effl0_aug', 'effl0_both', 'effl0_fine10']
rows, base = [], None
for n in names:
    p = f'{RUNS}/{n}/metrics.json'
    if not os.path.exists(p):
        rows.append((n, None)); continue
    m = json.load(open(p)); pc = m['test_f1_per_class']
    r = dict(val=m['best_val_macro_f1'], acc=m['test_acc'], f1=m['test_macro_f1'], normal=pc['normal'], talk=pc['talk_passenger'])
    r['gap'] = r['val'] - r['f1']
    if n == 'effl0_clean': base = r['f1']
    rows.append((n, r))
print(f"{'run':16s}{'val':>7s}{'test acc':>10s}{'test F1':>9s}{'vs mốc':>9s}{'normal':>8s}{'talk':>7s}{'val-test':>9s}")
for n, r in rows:
    if r is None: print(f'{n:16s}  THIẾU'); continue
    d = '' if base is None or n == 'effl0_clean' else f"{r['f1'] - base:+.3f}"
    print(f"{n:16s}{r['val']:7.3f}{r['acc']:10.3f}{r['f1']:9.3f}{d:>9s}{r['normal']:8.3f}{r['talk']:7.3f}{r['gap']:9.3f}")

### Cell 16: F1 trên 10 lớp gốc của `effl0_fine10`
Cell này chỉ có ý nghĩa với run `effl0_fine10`. Nó cho thấy trong 10 lớp gốc **lớp nào mô hình phân biệt tốt hay kém** (ví dụ c9 là `talk_passenger`; c1 và c3 là nhắn tin tay phải và tay trái). Thông tin này bị che mất khi chỉ nhìn 6 nhóm.

In [ ]:
import json
p = '/content/dds-edge/runs/effl0_fine10/metrics.json'
m = json.load(open(p))
print('Test F1 trên 10 lớp gốc:')
for k, v in m['test_f1_per_class_fine'].items():
    print(f'  {k}: {v:.3f}')
print('macro-F1 10 lớp:', round(m['test_macro_f1_fine'], 4), ' | macro-F1 6 nhóm:', round(m['test_macro_f1'], 4))

### Cell 17: confusion matrix của run tốt nhất
Đặt `BEST` là tên run có điểm cao nhất ở cell 15. In confusion matrix, F1 từng lớp, tài xế nào khó và các nhầm lẫn nhiều nhất. **Câu hỏi chính:** `talk_passenger` còn bị đoán thành `reach_grooming` nhiều như trước (56%) không? Nếu vẫn vậy thì nguyên nhân nằm ở ảnh nhập nhằng chứ không phải số ảnh mỗi lớp.

In [ ]:
BEST = 'effl0_fine10'   # đổi thành run tốt nhất ở cell 15
%cd /content/dds-edge
!python -m dds_edge.dl.analyze --run {BEST}

## F. Cách kết luận (đọc trước khi viết vào báo cáo)
1. **So với mốc 0,815**, cách nào vượt rõ (trên khoảng +0,03) thì mới coi là cải thiện thật. Nếu mọi cách đều trong ±0,03 thì kết luận trung thực là: *"xử lý mất cân bằng ở tầng dữ liệu không cải thiện vượt mức nhiễu"*, và điều đó **vẫn là một kết quả hợp lệ**.
2. Nhìn **F1 của `talk_passenger` và `normal`** chứ không chỉ macro-F1: một cách có thể tăng `talk_passenger` nhưng làm `phone` giảm (đổi chỗ chứ không tốt hơn).
3. Nhìn **val − test**: nếu một cách có val tăng mạnh mà test không tăng thì đó là dấu hiệu val bị lạc quan (đã gặp với `class_weight`).
4. Mỗi cấu hình mới chỉ chạy **một lần một seed**. Muốn chắc hơn, chạy lại cách tốt nhất với `--seed 1` và `--seed 2` rồi lấy trung bình.
5. Kết quả (`runs/`) đã nằm trong Drive: `MyDrive/dds-edge-runs/`. Tải các thư mục `effl0_cap`, `effl0_aug`, `effl0_both`, `effl0_fine10` về máy rồi báo để cập nhật runlog, slide và bài báo.